# Data recollection 
Data recollection for the found topics

In [ ]:
from google.colab import drive
drive.mount('/content/drive')   # authorize once



*   Collect for 2025
*   Collect for the not enough keywords again.
*   Delete from year data the keywords that you recollected.
*   Write a pipeline that concantates and clears the videos



In [ ]:
path = '/content/drive/MyDrive/Keyword-analysis/topics'

In [ ]:
#API keys:
FIRST_API = ''
SECOND_API = ''
THIRD_API = ''
FOURTH_API = ''
FIFTH_API = ''
SIXTH_API = ''
SEVENTH_API = ''
EIGTH_API = ''
NINTH_API = ''
TENTH_API = ''


API_11 = ''
API_12 = ''
API_13 = ''
API_14 = ''

In [ ]:
df.groupby('keywords').size()

In [ ]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
import pandas as pd
import time
import json

API_KEYS = [FIRST_API, SECOND_API, THIRD_API, FOURTH_API, FIFTH_API, SIXTH_API, SEVENTH_API, EIGTH_API, NINTH_API, TENTH_API, API_11, API_12, API_13, API_14]  # strings!

# "centenarians", "exercise & physical activity", "gut microbiome", "metformin", "rapamycin and sirolimus", "cellular senescence & senolytics", "telomere biology", "epigenetic and methylation",
#                  "mitochondria & mitophagy", "sirtuins", "autophagy", "diabetes & glycemic control", "cardiovascular disease", "obesity",
#                  "cancer", "modafinil and nootropics", "dementia and alzheimer"

#"nootropics, nootropics_2", "cellular senescence", "epigenetics", "mitochondria", "rapamycin"
SEARCH_TERMS = ["epigenetics", "mitochondria", "rapamycin"]
GENERAL_TERM = " and longevity and healthy aging"


# --- helpers ---
def get_youtube(api_key):
    return build("youtube", "v3", developerKey=api_key)

def is_quota_error(e: HttpError) -> bool:
    try:
        payload = json.loads(e.content.decode("utf-8"))
        reasons = [err.get("reason", "") for err in payload.get("error", {}).get("errors", [])]
        message = payload.get("error", {}).get("message", "")
        return any(r in ("quotaExceeded", "dailyLimitExceeded") for r in reasons) or "quota" in message.lower()
    except Exception:
        return "quota" in str(e).lower()

def with_key_rotation(call_fn, *args, **kwargs):
    """
    Run a Google API call function and rotate API keys if quota is exceeded.
    `call_fn` should be a lambda that uses the global `youtube`.
    """
    global key_index, youtube
    while True:
        try:
            return call_fn(*args, **kwargs)
        except HttpError as e:
            if is_quota_error(e):
                key_index += 1
                if key_index >= len(API_KEYS):
                    raise RuntimeError("All API keys exhausted!") from e
                print(f"?? Quota exceeded. Switching to next API key ({key_index+1}/{len(API_KEYS)})")
                youtube = get_youtube(API_KEYS[key_index])
                # retry immediately with new key
                continue
            else:
                # Re-raise non-quota errors to be handled by caller if needed
                raise

def get_all_videos(query, year, max_pages=5):
    """Fetch up to max_pages of search results (50 each)."""
    nx_year = year + 1
    all_results = []
    next_page_token = None
    for _ in range(max_pages):
        resp = with_key_rotation(
            lambda: youtube.search().list(
                q=query,
                part="snippet",
                publishedAfter=f"{year}-01-01T00:00:00Z",
                publishedBefore=f"{nx_year}-01-01T00:00:00Z",
                maxResults=50,
                type="video",
                relevanceLanguage="en",
                pageToken=next_page_token
            ).execute()
        )
        all_results.extend(resp.get("items", []))
        next_page_token = resp.get("nextPageToken")
        if not next_page_token:
            break
    return all_results

def chunked(iterable, size):
    buf = []
    for x in iterable:
        buf.append(x)
        if len(buf) == size:
            yield buf
            buf = []
    if buf:
        yield buf

In [ ]:
# --- main ---
key_index = 0
youtube = get_youtube(API_KEYS[key_index])

for term in SEARCH_TERMS:  # 2010?2025 inclusive
    video_data = []  # reset per year

    for year in range(2010, 2026):
        # 1) search (break after success)
        while True:
            try:
                search_term = term + GENERAL_TERM
                print(f"Searching with the term '{search_term}' for the year {year}")
                search_items = get_all_videos(search_term, year=year, max_pages=5)  # up to 250 per keyword
                break
            except HttpError as e:
                # non-quota HTTP error in search ? log and skip this term
                print("HTTP error during search:", e)
                search_items = []
                break

        if not search_items:
            continue

        # 2) collect IDs and batch-fetch video details
        ids = [it.get("id", {}).get("videoId") for it in search_items if it.get("id", {}).get("videoId")]
        details_by_id = {}

        for id_chunk in chunked(ids, 50):
            try:
                v_resp = with_key_rotation(
                    lambda: youtube.videos().list(
                        part="statistics,snippet",
                        id=",".join(id_chunk)
                    ).execute()
                )
            except HttpError as e:
                print("HTTP error during videos.list:", e)
                continue

            for item in v_resp.get("items", []):
                details_by_id[item["id"]] = item

            time.sleep(0.2)  # gentle pacing

        # 3) assemble rows + (optional) top comments with key rotation
        for it in search_items:
            vid = it.get("id", {}).get("videoId")
            if not vid:
                continue

            detail = details_by_id.get(vid)
            if not detail:
                # video may be deleted/private or filtered out
                continue

            snippet = detail.get("snippet", {}) or {}
            stats = detail.get("statistics", {}) or {}
            lang = snippet.get("defaultAudioLanguage") or snippet.get("defaultLanguage")

            # language filter: skip non-English if language is known
            if lang and not str(lang).lower().startswith("en"):
                continue

            # try to fetch a few top-level comments (often fails / disabled)
            comment_text = ""
            try:
                c_resp = with_key_rotation(
                    lambda: youtube.commentThreads().list(
                        part="snippet",
                        videoId=vid,
                        maxResults=5,
                        textFormat="plainText"
                    ).execute()
                )
                comments = [
                    c["snippet"]["topLevelComment"]["snippet"]["textDisplay"]
                    for c in c_resp.get("items", [])
                ]
                comment_text = " || ".join(comments)
            except HttpError:
                pass

            video_data.append({
                "source": "YouTube",
                "video_id": vid,
                "video_url": f"https://www.youtube.com/watch?v={vid}",
                "title": snippet.get("title") or it["snippet"]["title"],
                "description": snippet.get("description") or it["snippet"].get("description", ""),
                "date": snippet.get("publishedAt") or it["snippet"].get("publishedAt"),
                "channel": snippet.get("channelTitle") or it["snippet"].get("channelTitle", ""),
                "comments": comment_text,
                "views": stats.get("viewCount"),
                "likes": stats.get("likeCount"),
                "comments_count": stats.get("commentCount"),
                "keywords": term,
                "country": lang if lang else "unknown",
            })
            time.sleep(0.1)  # gentle pacing

    # 4) save ONCE per year (after all keywords)
    df = pd.DataFrame(video_data)
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df["year"] = year  # force the search year
        df["date"] = df["date"].dt.date
        # # optional: drop dupes across overlapping keywords
        # df = df.drop_duplicates(subset="video_id", keep="first")


    df.to_csv(f"{path}/{term}.csv", index=False)
    print(f"Done! Collected {len(df)} unique videos for term {term}")
    # no need to clear; it will be reinitialized next loop

# Data fix    


In [ ]:
#collecting year columns
import pandas as pd
dfs = []
for year in range (2010, 2026):
  df = pd.read_csv(f'{path}/{year}.csv')
  dfs.append(df)


In [ ]:
import pandas as pd
df = pd.concat(dfs, ignore_index=True)
df.shape

In [ ]:
#collecting keyword columns
terms = ["nootropics", "nootropics_2", "cellular senescence", "epigenetics", "mitochondria", "rapamycin"]
df_terms = []
for keyword in terms:
  df = pd.read_csv(f"{path}/{keyword}.csv")
  df = df.drop(columns=['year'])
  df["date"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
  df["year"] = df["date"].dt.year       # extract year while it's datetime
  df["date"] = df["date"].dt.date
  df_terms.append(df)

In [ ]:
dfk = pd.concat(df_terms, ignore_index=True)
dfk.shape

In [ ]:
dfk.groupby('keywords').size()

In [ ]:
df.groupby('keywords').size()

In [ ]:
#from df: delete - "cellular senescence & senolytics", "epigenetic and methylation", "mitochondria & mitophagy"
delete_terms = ["cellular senescence & senolytics", "epigenetic and methylation", "mitochondria & mitophagy", "rapamycin and sirolimus"]
df = df[~df['keywords'].isin(delete_terms)]
df.groupby('keywords').size()

In [ ]:
#in dfk change following:
rename_map = {
    "cellular senescence": "cellular senescence & senolytics",
    "epigenetics": "epigenetic and methylation",
    "mitochondria": "mitochondria & mitophagy",
    "nootropics": "modafinil and nootropics",
    "rapamycin": "rapamycin and sirolimus"
}


dfk["keywords"] = dfk["keywords"].replace(rename_map)
dfk.groupby('keywords').size()

In [ ]:
df_combined = pd.concat([df, dfk], ignore_index=True)
df_combined.groupby('keywords').size()

In [ ]:
df_combined.columns

In [ ]:
pd.set_option('display.max_rows', None)

In [ ]:
df_combined.groupby(['year', 'keywords']).size()

In [ ]:
df_combined.to_csv(f'{path}/data.csv', index=False)

## Data preprocessing

In [ ]:
import pandas as pd
df_youtube = pd.read_csv("/content/drive/MyDrive/Keyword-analysis/clean_YouTube.csv")

In [ ]:
df_youtube.columns

In [ ]:
import pandas as pd
df = pd.read_csv(f'{path}/data.csv')
df.columns

In [ ]:
print(df.groupby('keywords').size())
print(df.groupby('year').size())

In [ ]:
df.groupby(['year', 'keywords']).size()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df[df['comments_count'].isnull()].isnull().sum()

In [ ]:
df['comments_count']=df['comments_count'].fillna(0)

In [ ]:
# check comments_count and comments correnspond
df[df['comments'].isnull()].groupby('comments_count').size()

In [ ]:
df[(df['comments'].isnull()) & (df['comments_count']==2)]

In [ ]:
#pipeline

def duplicated_deal(df):
  print("Number of duplicates")
  print(df['video_id'].duplicated().sum())
  df = df.drop_duplicates(subset="video_id", keep="last")
  print("After cleaning")
  print(df['video_id'].duplicated().sum())
  return df



def null_values(df):
  df['description'] = df['description'].fillna('')
  df['comments']=df['comments'].fillna('')
  df = df[~df['views'].isnull()]
  df['likes_missing'] = df['likes'].isnull()
  df['likes'] = df['likes'].fillna(0)
  return df


In [ ]:
df = duplicated_deal(df)

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum()

In [ ]:
df = null_values(df)

In [ ]:
df.isnull().sum()

In [ ]:
df[df['likes_missing']==True].shape

In [ ]:
pd.set_option('display.max_rows', None)
df.groupby(['year', 'keywords']).size()

In [ ]:
df.groupby('keywords').size()

In [ ]:
# your dict: id -> name
topic_names = {
    0: "centenarians",
    1: "cellular senescence & senolytics",
    2: "telomere biology",
    3: "rapamycin and sirolimus",
    4: "metformin",
    5: "dementia and alzheimer",
    6: "exercise & physical activity",
    7: "diabetes & glycemic control",
    8: "gut microbiome",
    9: "epigenetic and methylation",
    10: "mitochondria & mitophagy",
    11: "sirtuins",
    12: "autophagy",
    13: "cardiovascular disease",
    14: "obesity",
    15: "cancer",
    16: "modafinil and nootropics",
}

# invert: name -> id
name_to_id = {v: k for k, v in topic_names.items()}

# exact match mapping (strings in df['keywords'])
df["topic_num"] = df["keywords"].map(name_to_id)  # unmatched => NaN

In [ ]:
df.isnull().sum()

In [ ]:
df.head()

In [ ]:
print(df.groupby('topic_num').size().sort_values())
print()
print(df.groupby('keywords').size().sort_values())

In [ ]:
df.shape

In [ ]:
df.duplicated().sum()

In [ ]:
df.to_csv(f'{path}/final_data.csv', index=False)